# Petit tour des salaires

On explore uniquement les données du secteur privé : salaires par secteur, sexe, taille d’entreprise et temps de travail, puis quelques indicateurs d’inégalités et leur évolution.

**Avant de commencer :** lance d’abord `Nettoyage_donnees.ipynb`. Ce notebook lit les CSV dans `Data nettoyees/` et ne modifie aucun fichier.

## 1. On charge les données

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def trouver_racine_projet():
    courant = Path.cwd().resolve()
    for dossier in (courant, *courant.parents):
        if (dossier / 'Data nettoyees').is_dir():
            return dossier
    raise FileNotFoundError(
        'Dossier « Data nettoyees » introuvable. Lance d’abord le notebook de nettoyage.'
    )


RACINE = trouver_racine_projet()
DOSSIER_DONNEES = RACINE / 'Data nettoyees'
CHEMIN_PRIVE = DOSSIER_DONNEES / 'DS_DERA_PRIVE_SERIES_LONGUES_data_nettoye.csv'
if not CHEMIN_PRIVE.is_file():
    raise FileNotFoundError(
        f'Fichier privé nettoyé absent : {CHEMIN_PRIVE}. '
        'Lance d’abord le notebook de nettoyage.'
    )

prive = pd.read_csv(CHEMIN_PRIVE, sep=';', encoding='utf-8-sig')
print(f"Données privées : {len(prive):,} lignes, de {prive['annee'].min()} à {prive['annee'].max()}.")

## 2. Quelques repères avant les graphiques

Les montants sont en **euros constants**, donc corrigés de l’inflation. Pour comparer une dimension (par exemple le sexe), on garde les autres dimensions sur « Ensemble ». Ça évite de comparer des groupes qui ne correspondent pas.

Les salaires sont des moyennes ou des quantiles publiés, pas des observations individuelles.

In [ ]:
SALAIRE = 'Salaire net moyen mensuel (Euros constants)'
MOYENNE = 'Ensemble'
P10 = '10e centile (D1 - 10% gagnent moins)'
P25 = '1er quartile (Q1 - 25% gagnent moins)'
P50 = 'Médiane (50% gagnent moins/plus)'
P90 = '90e centile (D9 - 10% gagnent plus)'
P95 = '95e centile (Top 5%)'
P99 = '99e centile (Top 1%)'

VALEURS_ENSEMBLE = {
    'secteur_activite': 'Ensemble des secteurs',
    'tranche_age': 'Ensemble',
    'taille_entreprise': 'Ensemble',
    'categorie_socio_prof': 'Ensemble',
    'sexe': 'Ensemble',
    'temps_travail': 'Ensemble',
}


def garder_ensemble_des_autres_dimensions(df, dimension=None):
    resultat = df.copy()
    for colonne, modalite in VALEURS_ENSEMBLE.items():
        if colonne in resultat.columns and colonne != dimension:
            resultat = resultat[resultat[colonne].eq(modalite)]
    return resultat


def extraire_salaire(df, quantiles, dimension=None):
    resultat = garder_ensemble_des_autres_dimensions(df, dimension)
    resultat = resultat[
        resultat['indicateur'].eq(SALAIRE)
        & resultat['unite'].eq('Euros constants')
        & resultat['centile_salaire'].isin(quantiles)
    ].copy()
    cles = ['annee', 'centile_salaire']
    if dimension:
        cles.insert(0, dimension)
    if resultat.duplicated(cles).any():
        raise ValueError(f'Plusieurs valeurs pour les mêmes clés : {cles}')
    return resultat


def derniere_annee_complete(table, groupe):
    annees_max = table.groupby(groupe)['annee'].max()
    return int(annees_max.min())


def tableau_salaire_recent(df, dimension):
    table = extraire_salaire(df, [MOYENNE], dimension)
    annee = derniere_annee_complete(table, dimension)
    table = table[table['annee'].eq(annee)].copy()
    table = table.rename(columns={dimension: 'Groupe', 'valeur': 'Salaire moyen (€ constants)'})
    return table[['Groupe', 'Salaire moyen (€ constants)']].sort_values(
        'Salaire moyen (€ constants)', ascending=False
    ), annee

## 3. Écart de salaire moyen entre hommes et femmes

On mesure l’écart avec **(salaire homme − salaire femme) / salaire homme × 100**. Par exemple, 10 % signifie que le salaire moyen des femmes est inférieur de 10 % à celui des hommes. On garde le secteur, l’âge, la taille d’entreprise, la catégorie socio-professionnelle et le temps de travail sur « Ensemble ».

In [ ]:
salaires_sexe = extraire_salaire(prive, [MOYENNE], 'sexe')
moyenne_sexe = salaires_sexe.pivot(
    index='annee', columns='sexe', values='valeur'
).dropna(subset=['Homme', 'Femme'])
moyenne_sexe['Écart H-F (% du salaire homme)'] = (
    (moyenne_sexe['Homme'] - moyenne_sexe['Femme']) / moyenne_sexe['Homme'] * 100
)

display(
    moyenne_sexe[['Homme', 'Femme', 'Écart H-F (% du salaire homme)']]
    .tail(10)
    .style.format({'Homme': '{:,.0f}', 'Femme': '{:,.0f}', 'Écart H-F (% du salaire homme)': '{:.1f}%'})
)

moyenne_sexe['Écart H-F (% du salaire homme)'].plot(figsize=(9, 4))
plt.title('Évolution de l’écart de salaire moyen — privé')
plt.xlabel('Année')
plt.ylabel('Écart par rapport au salaire moyen des hommes (%)')
plt.tight_layout()
plt.show()

print(
    f"Écart : {moyenne_sexe['Écart H-F (% du salaire homme)'].iloc[0]:.1f}% "
    f"en {moyenne_sexe.index.min()}, puis "
    f"{moyenne_sexe['Écart H-F (% du salaire homme)'].iloc[-1]:.1f}% "
    f"en {moyenne_sexe.index.max()}."
)

## 4. Ratio P90/P10 selon le sexe

Pour chaque sexe, on divise le salaire au 90e centile par celui du 10e. Un ratio de 3 signifie que le P90 est trois fois le P10. C’est un indicateur simple de dispersion, pas une mesure complète des inégalités.

On trace l’évolution des deux ratios et on les compare sur les mêmes années.

In [ ]:
def calculer_ratio_90_10(df, dimension=None):
    table = extraire_salaire(df, [P10, P90], dimension)
    index = ['annee'] if dimension is None else [dimension, 'annee']
    large = table.pivot(index=index, columns='centile_salaire', values='valeur')
    large = large.dropna(subset=[P10, P90]).copy()
    large['Ratio P90/P10'] = large[P90] / large[P10]
    return large.reset_index()


ratios_sexe = calculer_ratio_90_10(prive, 'sexe')
ratios_sexe = ratios_sexe[ratios_sexe['sexe'].isin(['Homme', 'Femme'])]
pivot_ratios_sexe = ratios_sexe.pivot(
    index='annee', columns='sexe', values='Ratio P90/P10'
).dropna(subset=['Homme', 'Femme'])

display(pivot_ratios_sexe.tail(10).style.format('{:.2f}'))
pivot_ratios_sexe.plot(figsize=(9, 4))
plt.title('Évolution du ratio P90/P10 selon le sexe — privé')
plt.xlabel('Année')
plt.ylabel('Ratio P90/P10')
plt.tight_layout()
plt.show()

print(
    f"Période commune : {pivot_ratios_sexe.index.min()}–{pivot_ratios_sexe.index.max()}. "
    f"En {pivot_ratios_sexe.index.max()}, le ratio est "
    f"{pivot_ratios_sexe['Homme'].iloc[-1]:.2f} pour les hommes et "
    f"{pivot_ratios_sexe['Femme'].iloc[-1]:.2f} pour les femmes."
)

## 5. Comparaison des P10 hommes et femmes

On compare **P10 homme / P10 femme** au fil du temps. Un ratio de 1 signifie que les deux P10 sont égaux; au-dessus de 1, le P10 des hommes est plus élevé.

In [ ]:
p10_sexe = extraire_salaire(prive, [P10], 'sexe')
pivot_p10_sexe = p10_sexe.pivot(
    index='annee', columns='sexe', values='valeur'
).dropna(subset=['Homme', 'Femme'])
pivot_p10_sexe['Ratio P10 homme/femme'] = (
    pivot_p10_sexe['Homme'] / pivot_p10_sexe['Femme']
)

display(
    pivot_p10_sexe[['Homme', 'Femme', 'Ratio P10 homme/femme']]
    .tail(10)
    .style.format({'Homme': '{:,.0f}', 'Femme': '{:,.0f}', 'Ratio P10 homme/femme': '{:.3f}'})
)
pivot_p10_sexe['Ratio P10 homme/femme'].plot(figsize=(9, 4))
plt.axhline(1, color='gray', linestyle='--')
plt.title('Évolution du ratio P10 homme / P10 femme — privé')
plt.xlabel('Année')
plt.ylabel('Ratio')
plt.tight_layout()
plt.show()

print(
    f"Le ratio passe de {pivot_p10_sexe['Ratio P10 homme/femme'].iloc[0]:.3f} "
    f"en {pivot_p10_sexe.index.min()} à "
    f"{pivot_p10_sexe['Ratio P10 homme/femme'].iloc[-1]:.3f} "
    f"en {pivot_p10_sexe.index.max()}."
)

## 6. Et du côté des hauts salaires ?

On suit P90, P95 et P99 en euros constants. On regarde aussi la part de la masse salariale reçue par les 10 %, 1 %, 0,1 % et 0,01 % les mieux rémunérés : ici, il s’agit d’une part en pourcentage, pas d’un salaire.

In [ ]:
def bilan_evolution(series):
    resultats = []
    for nom, groupe in series.groupby('Série'):
        groupe = groupe.sort_values('annee')
        debut, fin = groupe.iloc[0], groupe.iloc[-1]
        resultats.append(
            {
                'Série': nom,
                'Période': f"{int(debut['annee'])}–{int(fin['annee'])}",
                'Variation (%)': (
                    fin['Salaire (€ constants)'] / debut['Salaire (€ constants)'] - 1
                ) * 100,
            }
        )
    return pd.DataFrame(resultats).set_index('Série')


def serie_quantile(df, quantile, etiquette):
    table = extraire_salaire(df, [quantile])
    table = table[['annee', 'valeur']].rename(columns={'valeur': 'Salaire (€ constants)'})
    table['Série'] = etiquette
    return table


series_hauts = pd.concat(
    [
        serie_quantile(prive, P90, 'P90'),
        serie_quantile(prive, P95, 'P95'),
        serie_quantile(prive, P99, 'P99'),
    ],
    ignore_index=True,
)
pivot_hauts = series_hauts.pivot(
    index='annee', columns='Série', values='Salaire (€ constants)'
)
display(pivot_hauts.tail(10).style.format('{:,.0f}'))
pivot_hauts.plot(figsize=(10, 5))
plt.title('Évolution des hauts salaires — privé')
plt.xlabel('Année')
plt.ylabel('Euros constants par mois')
plt.tight_layout()
plt.show()

part_top = garder_ensemble_des_autres_dimensions(prive)
part_top = part_top[
    part_top['indicateur'].str.startswith('Part de la masse salariale perçue par le top ')
    & part_top['centile_salaire'].eq('Ensemble')
].copy()
part_top['Groupe'] = part_top['indicateur'].str.extract(r'(top .+)$', expand=False)
part_top = part_top.pivot(index='annee', columns='Groupe', values='valeur')
display(part_top.tail(10).style.format('{:.2f}%'))
part_top.plot(figsize=(10, 5))
plt.title('Part de la masse salariale détenue par le haut de la distribution — privé')
plt.xlabel('Année')
plt.ylabel('Part (%)')
plt.tight_layout()
plt.show()

## 7. Et du côté des salaires les plus bas ?

On suit P10 et P25 avec la médiane comme repère. Comme pour les salaires du haut, on compare les euros constants et on affiche les dates réellement disponibles.

In [ ]:
series_bas = pd.concat(
    [
        serie_quantile(prive, P10, 'P10'),
        serie_quantile(prive, P25, 'P25'),
        serie_quantile(prive, P50, 'Médiane'),
    ],
    ignore_index=True,
)

pivot_bas = series_bas.pivot(index='annee', columns='Série', values='Salaire (€ constants)')
display(pivot_bas.tail(10).style.format('{:,.0f}'))
pivot_bas.plot(figsize=(10, 5))
plt.title('Évolution des bas salaires — privé')
plt.xlabel('Année')
plt.ylabel('Euros constants par mois')
plt.tight_layout()
plt.show()

display(bilan_evolution(series_bas).style.format({'Début (€)': '{:,.0f}', 'Fin (€)': '{:,.0f}', 'Variation (%)': '{:+.1f}%'}))

## À garder en tête

- Les séries par sexe comparées ici couvrent les années où les deux valeurs sont disponibles; pour les données actuelles, elles vont de 1996 à 2024.
- Les périodes ne sont pas toujours les mêmes et certaines années peuvent manquer. Les tableaux d’évolution indiquent les années de début et de fin effectivement observées.
- Le ratio P90/P10 résume un écart entre deux points de la distribution; il ne raconte pas toute l’histoire des inégalités.
- Ces données décrivent des salaires mensuels en équivalent temps plein (EQTP); elles ne mesurent pas directement le revenu annuel ou le niveau de vie.